# llm

> Thin structured-output helper for OpenAI + Pydantic

In [ ]:
#| default_exp llm

## Design

This helper removes the duplicated OpenAI structured-output boilerplate used by `toc.py` and `summarize.py`.

It intentionally supports only the current shape: one prompt in, one structured Pydantic result out.

In [ ]:
#| export
import json
from typing import Any, TypeVar
import openai
from pydantic import BaseModel

TModel = TypeVar('TModel', bound=BaseModel)


In [ ]:
#| export
def generate_structured(prompt: str, # Full user prompt
                        response_model: type[TModel], # Pydantic response model
                        *,
                        schema_name: str, # JSON schema name for OpenAI response_format
                        model: str = 'gpt-5.4', # OpenAI model name
                        client: Any = None, # Optional prebuilt OpenAI-compatible client for tests
                       ) -> TModel: # Parsed response model instance
    "Call OpenAI structured output and validate into the caller-provided Pydantic model."
    client = client or openai.OpenAI()
    response = client.chat.completions.create(
        model=model,
        response_format={
            'type': 'json_schema',
            'json_schema': {
                'name': schema_name,
                'schema': response_model.model_json_schema(),
            },
        },
        messages=[{'role': 'user', 'content': prompt}],
    )
    # Some models occasionally emit trailing content after the primary JSON
    # object; extract just the first JSON value.
    content = response.choices[0].message.content
    obj, _ = json.JSONDecoder().raw_decode(content.lstrip())
    return response_model.model_validate(obj)


## Tests

In [ ]:
from pydantic import BaseModel

class _ToyResult(BaseModel):
    answer: str

class _FakeCompletions:
    def __init__(self, content='{"answer": "ok"}'):
        self.kw = None
        self.content = content
    def create(self, **kw):
        self.kw = kw
        content = self.content
        class _Message:
            pass
        _Message.content = content
        class _Choice:
            message = _Message()
        class _Response:
            choices = [_Choice()]
        return _Response()

class _FakeChat:
    def __init__(self, content='{"answer": "ok"}'):
        self.completions = _FakeCompletions(content)

class _FakeClient:
    def __init__(self, content='{"answer": "ok"}'):
        self.chat = _FakeChat(content)

# Happy path
client = _FakeClient()
result = generate_structured('hello', _ToyResult, schema_name='toy', model='gpt-test', client=client)
assert result == _ToyResult(answer='ok')
assert client.chat.completions.kw['model'] == 'gpt-test'
assert client.chat.completions.kw['messages'] == [{'role': 'user', 'content': 'hello'}]
assert client.chat.completions.kw['response_format']['json_schema']['name'] == 'toy'

# Trailing content after the first JSON object is tolerated.
client = _FakeClient(content='{"answer": "ok"}\n{"junk": 1}')
assert generate_structured('p', _ToyResult, schema_name='toy', client=client) == _ToyResult(answer='ok')
print('ok')
